# 16 — Ingestão da base real (`montar_base`)

Desenvolve o passo que baixa Ibovespa+CDI e popula o SQLite. **F1, NF6.**

In [ ]:
import sys, os, tempfile
_cwd = os.getcwd()
RAIZ = os.path.dirname(_cwd) if os.path.basename(_cwd) == 'tests' else _cwd
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)
import os
from app import dal

## Desenvolvimento

A(s) função(ões)/classe(s) abaixo foi(ram) escrita(s) aqui e, após os testes, movida(s) para `app/ingestao.py`.

In [ ]:
def montar_base(db_path: str = "data/mercado.db",
                inicio: str = "2000-01-01", 
                fim: str | None = None) -> dict:
    """Baixa Ibovespa (Yahoo) + CDI (Banco Central) e grava o SQLite.

    Returns
    -------
    dict com ``db_path``, ``n_meses`` e ``periodo`` (primeiro, último mês).
    """
    os.makedirs(os.path.dirname(db_path) or ".", exist_ok=True)

    # 1. Ibovespa — níveis mensais (tabela 'ibovespa').
    precos = dal.baixar_precos(["^BVSP"], inicio, fim)
    precos = precos.rename(columns={precos.columns[1]: "fechamento"})
    dal.gravar_sqlite(precos, db_path, "ibovespa")

    # 2. CDI — taxa mensal (tabela 'cdi').
    cdi = dal.baixar_cdi_bcb(inicio, fim)
    dal.gravar_sqlite(cdi, db_path, "cdi")

    # 3. Retornos alinhados por data (tabela 'retornos').
    ret_ibov = dal.calcular_retornos(precos.rename(columns={"fechamento": "ibov"}))
    retornos = ret_ibov.merge(cdi, on="data", how="inner")
    if retornos.empty:
        raise ValueError("Sem datas em comum entre Ibovespa e CDI — verifique o período.")
    dal.gravar_sqlite(retornos, db_path, "retornos")

    return {"db_path": db_path, 
            "n_meses": len(retornos),
            "periodo": (retornos["data"].iloc[0], retornos["data"].iloc[-1])}


**Teste** — monta o banco (download real, protegido).

In [25]:
import tempfile
try:
    dbm = os.path.join(tempfile.gettempdir(), 'dev16.db')
    info = montar_base(dbm, '2020-01-01', '2020-06-01')
    print('montar_base ->', info); print(dal.ler_sqlite(dbm,'retornos'))
    assert set(dal.ler_sqlite(dbm,'retornos').columns) == {'data','ibov','cdi'}
    os.remove(dbm); print('ingestao: PASSOU')
except Exception as e:
    print('offline:', type(e).__name__, e)

montar_base -> {'db_path': 'C:\\Users\\MURILO\\AppData\\Local\\Temp\\dev16.db', 'n_meses': 4, 'periodo': ('2020-02', '2020-05')}
      data      ibov     cdi
0  2020-02 -0.084291  0.0029
1  2020-03 -0.299044  0.0034
2  2020-04  0.102520  0.0028
3  2020-05  0.085671  0.0024
ingestao: PASSOU


✔ **Testado e aprovado — código movido para `app/ingestao.py`.**